# Horse Racing RL — Phase 2: Archetype Specialization

Fine-tune per-horse jockey policies from Phase 1 baseline ONNX model.

**Prerequisites:** Upload your `baseline_jockey.onnx` from Phase 1 training.

**Archetypes** (derived from 79,447 HKJC race runs):
- **Front-runner** — fast start, lead throughout (12.4% win rate)
- **Stalker** — sits near front, moves to lead late (13.0% win rate)
- **Closer** — conserves energy, big late move (3.2% win rate)
- **Presser** — pushes pace consistently, wears down field (3.2% win rate)

**Outputs:** 4 per-archetype ONNX models for browser inference.

## 1. Setup

In [ ]:
import os

# Clone or pull latest
if os.path.exists("hr-simulation"):
    !cd hr-simulation && git pull
else:
    !git clone https://github.com/ue-too/hr-simulation.git

%cd hr-simulation

In [ ]:
# Remove old gym (Colab pre-installs it) and install dependencies
!pip uninstall -y gym 2>/dev/null
!pip install -q stable-baselines3 gymnasium pettingzoo numpy torch onnx onnxruntime tensorboard

In [ ]:
# Verify imports
from horse_racing.engine import HorseRacingEngine, EngineConfig
from horse_racing.types import HorseAction, HORSE_COUNT, OBS_SIZE
from horse_racing.env import HorseRacingSingleEnv
from horse_racing.reward import compute_reward, ARCHETYPES
print(f"Engine OK — {HORSE_COUNT} horses, OBS_SIZE={OBS_SIZE}")
print(f"Archetypes: {ARCHETYPES}")

## 2. Load Phase 1 Baseline

Upload your `baseline_jockey.onnx` from Phase 1, then convert it to SB3 weights for fine-tuning.

In [ ]:
import onnx
import onnx.numpy_helper
import torch
import numpy as np
from pathlib import Path
from stable_baselines3 import PPO

# Upload ONNX file
ONNX_PATH = "baseline_jockey.onnx"

if not Path(ONNX_PATH).exists():
    try:
        from google.colab import files
        print("Upload your baseline_jockey.onnx file:")
        uploaded = files.upload()
        ONNX_PATH = list(uploaded.keys())[0]
    except ImportError:
        raise FileNotFoundError(f"{ONNX_PATH} not found. Place it in the working directory.")

# Load ONNX weights
onnx_model = onnx.load(ONNX_PATH)
onnx_weights = {}
for init in onnx_model.graph.initializer:
    onnx_weights[init.name] = torch.tensor(np.array(onnx.numpy_helper.to_array(init)))

print(f"ONNX loaded: {ONNX_PATH}")
for k, v in onnx_weights.items():
    print(f"  {k}: {list(v.shape)}")

# Determine architecture from weight shapes
layer0_in = onnx_weights["mlp_extractor.policy_net.0.weight"].shape[1]
layer0_out = onnx_weights["mlp_extractor.policy_net.0.weight"].shape[0]
layer1_out = onnx_weights["mlp_extractor.policy_net.2.weight"].shape[0]
print(f"\nArchitecture: {layer0_in} → {layer0_out} → {layer1_out} → 2")
assert layer0_in == OBS_SIZE, f"ONNX input dim {layer0_in} != OBS_SIZE {OBS_SIZE}"

# Create SB3 model with matching architecture and inject weights
dummy_env = HorseRacingSingleEnv(track_path="tracks/tokyo.json", max_steps=5000)
base_model = PPO(
    "MlpPolicy", dummy_env, verbose=0,
    policy_kwargs={"net_arch": [layer0_out, layer1_out]},
    device="cpu",
)
dummy_env.close()

state_dict = base_model.policy.state_dict()
for name in onnx_weights:
    if name in state_dict:
        state_dict[name] = onnx_weights[name]
base_model.policy.load_state_dict(state_dict)

print(f"\nPhase 1 weights loaded into SB3 model")
print(f"Policy network: {sum(p.numel() for p in base_model.policy.parameters()):,} parameters")

## 3. Archetype Environment

Wraps the single-agent env to inject archetype-specific reward shaping.

In [ ]:
import gymnasium as gym
import numpy as np
import random

# Amplify archetype reward signal so it's audible vs the dominant progress reward.
# Archetype bonuses are ~0.1-0.5/tick, base progress reward is ~2-5/tick.
# At 5x, archetype signal becomes ~0.5-2.5/tick — meaningful but not dominant.
ARCHETYPE_SCALE = 5.0


class ArchetypeEnv(gym.Wrapper):
    """Wraps HorseRacingSingleEnv with amplified archetype reward shaping.

    Computes reward with and without archetype, then scales the archetype-
    specific component to make it audible against the base progress reward.
    Rotates tracks on each reset for better generalization.
    """

    def __init__(self, tracks, max_steps, archetype):
        self.tracks = tracks if isinstance(tracks, list) else [tracks]
        self.max_steps = max_steps
        self.archetype = archetype
        env = HorseRacingSingleEnv(
            track_path=random.choice(self.tracks), max_steps=max_steps
        )
        super().__init__(env)

    def step(self, action):
        obs_array, _, terminated, truncated, info = self.env.step(action)

        obs_curr = self.env.engine.get_observations()[0]
        placements = self.env.engine.get_placements()
        finish_order = placements[0] if obs_curr["finished"] else None

        reward_args = dict(
            obs_prev=self._prev_obs_for_reward,
            obs_curr=obs_curr,
            collision_occurred=obs_curr["collision"],
            placement=placements[0],
            num_horses=self.env.engine.horse_count,
            finish_order=finish_order,
            prev_placement=self._prev_placement,
        )

        # Compute base reward (no archetype) and full reward (with archetype)
        base_reward = compute_reward(**reward_args, archetype=None)
        full_reward = compute_reward(**reward_args, archetype=self.archetype)
        archetype_bonus = full_reward - base_reward

        # Amplify archetype signal
        reward = base_reward + ARCHETYPE_SCALE * archetype_bonus

        self._prev_placement = placements[0]
        self._prev_obs_for_reward = obs_curr

        return obs_array, reward, terminated, truncated, info

    def reset(self, **kwargs):
        new_track = random.choice(self.tracks)
        self.env = HorseRacingSingleEnv(
            track_path=new_track, max_steps=self.max_steps
        )
        obs, info = self.env.reset(**kwargs)
        self._prev_obs_for_reward = self.env.engine.get_observations()[0]
        self._prev_placement = 1
        return obs, info


print(f"ArchetypeEnv ready (ARCHETYPE_SCALE={ARCHETYPE_SCALE}x)")

## 4. Training Configuration

In [ ]:
DEVICE = "cpu"
N_ENVS = 32              # scaled up from 8 — Colab Pro has ~50GB RAM
BATCH_SIZE = 1024         # scaled up from 256
TIMESTEPS_PER_ARCHETYPE = 2_000_000  # 4x budget for meaningful differentiation

LOG_DIR = "logs/archetypes"

# Two-phase schedule per archetype:
# Stage 1: high LR + entropy to break out of baseline behavior
# Stage 2: low LR + entropy to refine and converge
PHASE_SCHEDULE = [
    {"timesteps": 1_500_000, "lr": 3e-4, "ent_coef": 0.01, "label": "explore"},
    {"timesteps":   500_000, "lr": 1e-4, "ent_coef": 0.003, "label": "refine"},
]

# Train on multiple tracks for generalization
TRAINING_TRACKS = [
    "tracks/curriculum_2_gentle_oval.json",
    "tracks/curriculum_3_tight_oval.json",
    "tracks/tokyo.json",
    "tracks/kokura.json",
    "tracks/hanshin.json",
    "tracks/kyoto.json",
    "tracks/tokyo_2600.json",
]

print(f"Archetypes: {ARCHETYPES}")
print(f"Timesteps per archetype: {TIMESTEPS_PER_ARCHETYPE:,}")
print(f"Total timesteps: {TIMESTEPS_PER_ARCHETYPE * len(ARCHETYPES):,}")
print(f"Training tracks: {len(TRAINING_TRACKS)}")
print(f"Envs per archetype: {N_ENVS}")
print(f"Schedule per archetype:")
for p in PHASE_SCHEDULE:
    print(f"  {p['label']}: {p['timesteps']:,} steps, lr={p['lr']}, ent={p['ent_coef']}")

## 5. Train Archetypes

In [ ]:
import time
import random
from stable_baselines3.common.vec_env import SubprocVecEnv, DummyVecEnv
from stable_baselines3.common.callbacks import BaseCallback


class ProgressCallback(BaseCallback):
    def __init__(self, stage_name, total_timesteps, arch_idx, num_archetypes, print_freq=2048):
        super().__init__()
        self.stage_name = stage_name
        self.total = total_timesteps
        self.print_freq = print_freq
        self.arch_idx = arch_idx
        self.num_archetypes = num_archetypes
        self.start_time = None

    def _on_training_start(self):
        self.start_time = time.time()

    def _on_step(self) -> bool:
        if self.n_calls % self.print_freq == 0:
            elapsed = time.time() - self.start_time
            steps = self.num_timesteps
            sps = steps / elapsed if elapsed > 0 else 0
            pct = 100 * steps / self.total
            overall = 100 * (self.arch_idx + steps / self.total) / self.num_archetypes
            eta = (self.total - steps) / sps if sps > 0 else 0

            if len(self.model.ep_info_buffer) > 0:
                mean_r = sum(ep['r'] for ep in self.model.ep_info_buffer) / len(self.model.ep_info_buffer)
                reward_str = f"reward: {mean_r:8.2f}"
            else:
                reward_str = "reward:      n/a"

            print(f"  [{self.stage_name}] {pct:5.1f}% | overall: {overall:4.1f}% | steps: {steps:>8,} | {reward_str} | {sps:.0f} sps | ETA: {eta/60:.1f}m")
        return True


def make_archetype_env(archetype, tracks, max_steps):
    """Create env factory with track rotation on each reset."""
    def _init():
        return ArchetypeEnv(tracks=tracks, max_steps=max_steps, archetype=archetype)
    return _init


checkpoint_dir = Path("checkpoints/archetypes")
checkpoint_dir.mkdir(parents=True, exist_ok=True)

VecEnvClass = SubprocVecEnv if N_ENVS > 1 else DummyVecEnv

archetype_models = {}
total_start = time.time()

for arch_idx, archetype in enumerate(ARCHETYPES):
    print(f"\n{'='*60}")
    print(f"[{arch_idx+1}/{len(ARCHETYPES)}] Training: {archetype} — {TIMESTEPS_PER_ARCHETYPE:,} timesteps")
    print(f"{'='*60}")

    env = VecEnvClass([make_archetype_env(archetype, TRAINING_TRACKS, 5000) for _ in range(N_ENVS)])

    # Initialize with Phase 1 weights + first phase hyperparameters
    phase = PHASE_SCHEDULE[0]
    model = PPO(
        "MlpPolicy", env, verbose=0,
        policy_kwargs={"net_arch": [layer0_out, layer1_out]},
        n_steps=2048, batch_size=BATCH_SIZE, n_epochs=10,
        learning_rate=phase["lr"], gamma=0.99, gae_lambda=0.95,
        clip_range=0.2, vf_coef=0.5, ent_coef=phase["ent_coef"],
        device=DEVICE,
        tensorboard_log=LOG_DIR,
    )

    # Copy Phase 1 weights
    model.policy.load_state_dict(base_model.policy.state_dict())
    print(f"  Initialized from Phase 1 baseline weights")

    # Two-phase training: explore then refine
    for phase_idx, phase in enumerate(PHASE_SCHEDULE):
        print(f"  Phase {phase_idx+1}/{len(PHASE_SCHEDULE)}: {phase['label']} — "
              f"{phase['timesteps']:,} steps, lr={phase['lr']}, ent={phase['ent_coef']}")

        # Update hyperparameters for this phase
        model.learning_rate = phase["lr"]
        model.ent_coef = phase["ent_coef"]

        callback = ProgressCallback(
            f"{archetype}/{phase['label']}", phase["timesteps"],
            arch_idx, len(ARCHETYPES),
        )
        model.learn(
            total_timesteps=phase["timesteps"], callback=callback,
            reset_num_timesteps=(phase_idx == 0),
            tb_log_name=archetype,
        )

    save_path = checkpoint_dir / f"jockey_{archetype}"
    model.save(str(save_path))
    archetype_models[archetype] = model
    print(f"  Saved → {save_path}")
    env.close()

total_time = time.time() - total_start
print(f"\nAll archetypes trained in {total_time/60:.1f} minutes")

## 6. Export to ONNX

In [ ]:
import torch
import torch.nn as nn
import onnxruntime as ort


class PolicyNetwork(nn.Module):
    def __init__(self, sb3_policy):
        super().__init__()
        self.features_extractor = sb3_policy.features_extractor
        self.mlp_extractor = sb3_policy.mlp_extractor
        self.action_net = sb3_policy.action_net

    def forward(self, obs):
        features = self.features_extractor(obs)
        latent_pi, _ = self.mlp_extractor(features)
        return self.action_net(latent_pi)


dummy = torch.zeros(1, OBS_SIZE, dtype=torch.float32)

for archetype, model in archetype_models.items():
    wrapper = PolicyNetwork(model.policy)
    wrapper.eval()

    onnx_path = str(checkpoint_dir / f"jockey_{archetype}.onnx")
    torch.onnx.export(
        wrapper, dummy, onnx_path,
        input_names=["obs"], output_names=["action"],
        dynamic_axes={"obs": {0: "batch"}, "action": {0: "batch"}},
        opset_version=17, dynamo=False,
    )

    # Verify
    sess = ort.InferenceSession(onnx_path)
    test = np.zeros((1, OBS_SIZE), dtype=np.float32)
    result = sess.run(["action"], {"obs": test})
    print(f"{archetype:15s} → {onnx_path}  (output: {result[0][0]})")

print(f"\n4 ONNX models exported to {checkpoint_dir}/")

## 7. Quick Evaluation — Head-to-Head Race

In [ ]:
# Run a 4-horse race with each archetype controlling one horse
engine = HorseRacingEngine("tracks/tokyo.json")

# Load all 4 ONNX sessions
sessions = {}
for arch in ARCHETYPES:
    onnx_path = str(checkpoint_dir / f"jockey_{arch}.onnx")
    sessions[arch] = ort.InferenceSession(onnx_path)

print(f"Racing: {' vs '.join(ARCHETYPES)}")
print(f"Track: Tokyo\n")

for tick in range(5000):
    all_obs = engine.get_observations()
    actions = []

    for i, arch in enumerate(ARCHETYPES):
        arr = engine.obs_to_array(all_obs[i]).reshape(1, -1)
        action = sessions[arch].run(["action"], {"obs": arr})[0][0]
        actions.append(HorseAction(float(action[0]), float(action[1])))

    engine.step(actions)

    if tick % 1000 == 999:
        placements = engine.get_placements()
        print(f"Tick {tick+1:4d}:")
        for i, arch in enumerate(ARCHETYPES):
            o = all_obs[i]
            print(
                f"  {arch:15s} | P{placements[i]} | "
                f"progress: {o['track_progress']:.3f} | "
                f"vel: {o['tangential_vel']:.1f} | "
                f"stamina: {o['stamina_ratio']:.3f}"
            )
        print()

# Final results
print(f"{'='*60}")
print("FINAL RESULTS")
print(f"{'='*60}")
placements = engine.get_placements()
final_obs = engine.get_observations()
for i, arch in enumerate(ARCHETYPES):
    o = final_obs[i]
    status = "FINISHED" if o["finished"] else f"{o['track_progress']:.1%}"
    print(f"  P{placements[i]}: {arch:15s} | {status} | stamina: {o['stamina_ratio']:.3f}")

## 8. Download Models

Download all 4 ONNX files for browser inference.

In [ ]:
try:
    from google.colab import files
    for arch in ARCHETYPES:
        onnx_path = str(checkpoint_dir / f"jockey_{arch}.onnx")
        files.download(onnx_path)
    print("Downloads started!")
except ImportError:
    print(f"Not in Colab. Models saved at: {checkpoint_dir}/jockey_*.onnx")

In [ ]:
# Download TensorBoard logs for local viewing
# Run locally: tensorboard --logdir logs/archetypes
import shutil
shutil.make_archive("tb_logs_archetypes", "zip", ".", LOG_DIR)
try:
    from google.colab import files
    files.download("tb_logs_archetypes.zip")
except ImportError:
    print(f"Not in Colab. Logs at: {LOG_DIR}/")